In [7]:
import os
import random
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from matplotlib.transforms import ScaledTranslation
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties

import seaborn as sns

# Add project source to path
sys.path.append(
    os.path.join(
        '..',  # notes -> root
    )
)

from src.simulation.kernels import simple_kernel
import fig_settings as fs

fs.set_colors()
fs.set_fonts()

In [2]:
def _hpdi(x, prob=0.5):
    """Narrowest interval containing `prob` of 1D samples."""
    x = np.sort(x)
    n = len(x)
    k = int(np.floor(prob * n))      # number of points inside the interval
    widths = x[k:] - x[:n - k]       # width of every candidate window
    i = np.argmin(widths)            # pick the narrowest
    return x[i], x[i + k]

def hpdi_bounds(X, prob=0.5):
    lower, upper = np.empty(X.shape[1]), np.empty(X.shape[1])
    for j in range(X.shape[1]):
        lower[j], upper[j] = _hpdi(X[:, j], prob=prob)
    return lower, upper

In [3]:
def boxed_text(ax, x, y, s, pad=4, fontsize=20, boxstyle='round',
               fc='white', ec='k', text_kw=None, box_kw=None):
    text_kw = text_kw or {}; box_kw = box_kw or {}
    fig = ax.figure
    t = ax.text(x, y, s, ha='center', va='center', fontsize=fontsize,
                transform=ax.transAxes, **text_kw)
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    scale = fig.dpi / 72.0

    layout  = t.get_window_extent(r)
    A = layout.width  / scale                         # advance width (points)
    H = layout.height / scale                         # layout height (points)
    fp = FontProperties(size=fontsize)
    ink     = TextPath((0, 0), s, prop=fp).get_extents()
    descent = -TextPath((0, 0), 'Ayg|', prop=fp).get_extents().y0

    x0, x1 = -A/2 + ink.x0, -A/2 + ink.x1             # ink box relative to anchor
    base   = -H/2 + descent                           # baseline relative to anchor
    y0, y1 = base + ink.y0, base + ink.y1

    trans = fig.dpi_scale_trans + ScaledTranslation(x, y, ax.transAxes)
    box = FancyBboxPatch(((x0 - pad)/72, (y0 - pad)/72),
                         (x1 - x0 + 2*pad)/72, (y1 - y0 + 2*pad)/72,
                         boxstyle=f'{boxstyle},pad=0', fc=fc, ec=ec,
                         transform=trans, zorder=t.get_zorder() - 1, **box_kw)
    ax.add_patch(box)
    return t, box

In [4]:
RESULTS_DIR = Path("../results/complexity/")

# load heatmap data
fpath = RESULTS_DIR / "epsilon_omega_sweep" / "complexity_by_setting.parquet"
df = pd.read_parquet(fpath)
df["coef_var"] = df["complexity_std"] / df["complexity_mean"] # Create coefficient of variation column
piv_a = df.pivot(index="omega", columns="epsilon", values="coef_var")

fpath = RESULTS_DIR / "beta_delta_sweep" / "complexity_by_setting.parquet"
df = pd.read_parquet(fpath)
df["coef_var"] = df["complexity_std"] / df["complexity_mean"] # Create coefficient of variation column
print(df.head())
piv_b = df.pivot(index="beta2", columns="delta", values="coef_var")

   beta2  delta  complexity_mean  complexity_std  n_reps  mean_distance  \
0   0.01 -0.900         0.415010        0.076935      10      54.976960   
1   0.01 -0.875         0.487263        0.110397      10      57.750254   
2   0.01 -0.850         0.447449        0.140265      10      55.754356   
3   0.01 -0.825         0.446443        0.115576      10      55.052606   
4   0.01 -0.800         0.471610        0.091393      10      55.631429   

   trace_p_sigma  lw_reg_eff  coef_var  
0      38.857165    0.018692  0.185381  
1      38.822584    0.018373  0.226567  
2      38.508244    0.016648  0.313476  
3      38.049999    0.019171  0.258883  
4      37.795672    0.017601  0.193789  


In [19]:
complexity_cmap = sns.color_palette("crest_r", as_cmap=True)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(4.5, 2.25), layout="constrained")

vmin = np.min([np.min(piv_a), np.min(piv_b)])
vmax = np.max([np.max(piv_a), np.max(piv_b)])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

#  >>> Panel a - epsilon/omega heatmap >>>
# Plot data
im = axs[0].imshow(piv_a, cmap=complexity_cmap, aspect="auto", norm=norm)

# Setup axes and labels
axs[0].set_xlabel("Community\nseparation, $\\varepsilon$")
axs[0].set_ylabel("Block-flip prob., $\\omega$")
axs[0].invert_yaxis()
axs[0].set_xticks(np.arange(len(piv_a.columns))[0::9])
axs[0].set_xticklabels([f"{b:.3g}" for b in piv_a.columns[0::9]])
axs[0].set_yticks(np.arange(len(piv_a.index))[::4])
axs[0].set_yticklabels([f"{d:.3g}" for d in piv_a.index[::4]])

# Plot reference points
# axs[0].scatter([17], [0], marker='o', s=15, fc=fs.palette[4], ec='black', lw=0.25, zorder=5)
# axs[0].scatter([17], [20], marker='o', s=15, fc=fs.palette[3], ec='black', lw=0.25, zorder=5)



# Za = piv_a.to_numpy()
# M, N = Za.shape
# xs = np.linspace(-0.5, N - 0.5, N)
# ys = np.linspace(-0.5, M - 0.5, M)
# levels = [1.0, 5.0, 10.0, 50.0, 100.0]
# cs = axs[0].contour(xs, ys, Za, levels=levels, colors='white', linewidths=0.6)
# labels = axs[0].clabel(cs, inline=True, fontsize=7, rightside_up=True)
# for t, deg in zip(labels, [30, 15, 30]):
#     t.set_rotation(deg)

# plt.colorbar(im, ax=axs[0], label="Complexity coefficient of variation", location='top', pad=0.02)
# <<< Panel a <<<


# >>> Panel b - beta/delta heatmap >>>
# Plot data
im = axs[1].imshow(piv_b, cmap=complexity_cmap, aspect="auto", norm=norm)

# Setup axes and labels
axs[1].set_xlabel("Density\nimbalance, $\\delta$")
axs[1].set_ylabel("Kernel contrast, $r_\\beta$")
axs[1].invert_yaxis()
axs[1].set_xticks(np.arange(len(piv_b.columns))[0::36])
axs[1].set_xticklabels([f"{b:.3g}" for b in piv_b.columns[0::36]])
axs[1].set_yticks(np.arange(len(piv_b.index))[::4])
axs[1].set_yticklabels([f"{d:.3g}" for d in piv_b.index[::4]/0.01])

# Plot reference points
# axs[1].scatter([36], [12], marker='X', s=30, fc='white', ec='black', lw=0.5, zorder=5)
# axs[1].scatter([10], [16], marker='o', s=15, fc=fs.palette[1], ec='black', lw=0.25, zorder=5)
# axs[1].scatter([62], [16], marker='o', s=15, fc=fs.palette[2], ec='black', lw=0.25, zorder=5)


# Zb = piv_b.to_numpy()
# M, N = Zb.shape
# xs = np.linspace(-0.5, N - 0.5, N)
# ys = np.linspace(-0.5, M - 0.5, M)
# levels = [5.0, 10.0, 50.0, 100.0]
# cs = axs[1].contour(xs, ys, Zb, levels=levels, colors='white', linewidths=0.6)
# manual_coords = [(10.0, 2.0), (5.0, 5.0), (20.0, 15.1), (70.0, 1.0), (55.5, 3.0), (70, 7.0), (65, 13.5)]
# labels = axs[1].clabel(cs, inline=True, fontsize=7, manual=manual_coords, rightside_up=True, inline_spacing=6)
# for t, deg in zip(labels, [330, 315, 285, 12, 7, 50, 75]):
#     t.set_rotation(deg)

plt.colorbar(im, ax=axs[1], label="Complexity RSD", location='right', pad=0.05)
# <<< Panel b <<<

# Panel labels
axs[0].text(-0.15, 1.1, "(a)", in_layout=True, transform=axs[0].transAxes, fontsize=8, fontweight='bold', va='center', ha='left')
axs[1].text(-0.15, 1.1, "(b)", in_layout=True, transform=axs[1].transAxes, fontsize=8, fontweight='bold', va='center', ha='left')

plt.savefig('../figs/figS1.svg', bbox_inches='tight')
plt.savefig('../figs/figS1.pdf', bbox_inches='tight')